# Chapter 32 — Cloud Deployment: Compute, Containers, Scaling & Serverless

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part5_production_rl/ch32_cloud_deployment/ch32_notebook.ipynb)

Companion notebook for **Chapter 32** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Match the compute to the workload: virtual machines for control, GPU and TPU nodes for training, serverless for light inference.
- Object storage suits datasets and artifacts, block storage suits active training, file storage suits distributed jobs.
- Containers bundle code, dependencies and runtime, which eliminates environment drift between laptop, CI and production.
- Horizontal scaling adds machines or pods; vertical scaling grows one machine and is capped by hardware.
- Spot and preemptible instances cut cost by 70 to 90 percent for fault-tolerant work, provided you checkpoint.
- Serverless cannot train large models: execution time, memory and the absence of GPUs are hard limits.

## Example 1: Autoscaling Simulation

This runs the Kubernetes HPA control loop over a realistic 24-hour traffic curve: monitor utilisation, compare with the target, and adjust replica count. The comparison at the end quantifies the trade-off - autoscaling holds latency within SLA at 55% of the cost of permanent over-provisioning, whereas fixed capacity is cheaper but breaches the SLA in 14 of 24 hours.

In [1]:
# Chapter 32 - Example 1: Autoscaling simulation (Kubernetes HPA style)
import numpy as np
np.random.seed(0)

print("=== Cloud Autoscaling for ML Inference ===\n")
print("Scalability = adjust capacity to match demand while keeping latency")
print("stable. The control loop:")
print("  monitor metrics -> evaluate thresholds -> adjust capacity -> stabilise\n")

TARGET_CPU   = 0.70     # target utilisation per replica
CAP_PER_POD  = 100      # requests/sec one replica can serve
MIN_PODS, MAX_PODS = 2, 20
BASE_LAT     = 40       # ms at low load

# A day of traffic: quiet night, morning ramp, lunch peak, evening spike
hours = np.arange(24)
traffic = np.array([120,90,70,60,70,110,220,420,650,780,830,910,
                    1180,980,860,820,900,1150,1320,1080,760,520,320,180])

def latency(load_ratio):
    """Latency grows sharply as utilisation approaches 1 (queueing)."""
    return BASE_LAT / max(1e-3, (1 - min(load_ratio, 0.99)))

print(f"Config: {CAP_PER_POD} req/s per replica, target CPU {TARGET_CPU:.0%}, "
      f"min={MIN_PODS}, max={MAX_PODS} replicas\n")
print(f"{'hour':>5}{'req/s':>8}{'replicas':>10}{'util':>8}{'latency':>10}  {'action'}")
print("-" * 62)

pods = MIN_PODS
fixed_pods = 6            # for the comparison below
total_pod_hours, sla_breaches = 0, 0
fixed_breaches = 0

for h, rps in zip(hours, traffic):
    # --- HPA formula: desired = ceil(current * (currentUtil / targetUtil)) ---
    util_now = rps / (pods * CAP_PER_POD)
    desired  = int(np.ceil(pods * (util_now / TARGET_CPU)))
    desired  = max(MIN_PODS, min(MAX_PODS, desired))
    action   = ("scale UP" if desired > pods else
                "scale DOWN" if desired < pods else "hold")
    pods = desired

    util = rps / (pods * CAP_PER_POD)
    lat  = latency(util)
    total_pod_hours += pods
    if lat > 200: sla_breaches += 1

    fixed_util = rps / (fixed_pods * CAP_PER_POD)
    if latency(fixed_util) > 200: fixed_breaches += 1

    if h % 3 == 0 or h in (12, 18):
        print(f"{h:>5}{rps:>8}{pods:>10}{util:>8.0%}{lat:>9.0f}ms  {action}")

# ---- Cost comparison ----
POD_HOUR = 0.10
print("\n=== Autoscaling vs Fixed Capacity (24 hours) ===\n")
fixed_hours = fixed_pods * 24
over_hours  = MAX_PODS * 24
print(f"{'Strategy':<34}{'pod-hours':>11}{'cost':>10}{'SLA breaches':>14}")
print("-" * 70)
print(f"  {'Autoscaling (2-20 replicas)':<32}{total_pod_hours:>11}${total_pod_hours*POD_HOUR:>9.2f}{sla_breaches:>14}")
print(f"  {'Fixed 6 replicas':<32}{fixed_hours:>11}${fixed_hours*POD_HOUR:>9.2f}{fixed_breaches:>14}")
print(f"  {'Over-provisioned 20 replicas':<32}{over_hours:>11}${over_hours*POD_HOUR:>9.2f}{0:>14}")

saving = (over_hours - total_pod_hours) / over_hours * 100
print(f"\nAutoscaling saves {saving:.0f}% versus permanent over-provisioning,")
print(f"while keeping SLA breaches at {sla_breaches} (fixed capacity: {fixed_breaches}).")

print("\n=== Types of autoscaling ===")
print("  Instance autoscaling : VM level - EC2 Auto Scaling Groups, GCE Instance Groups")
print("  Container autoscaling: pod level - Kubernetes HPA (horizontal) and VPA (vertical)")
print("  Serverless autoscaling: function level - scales to ZERO when idle, pay per call")
print("\nHorizontal scaling adds machines/pods; vertical scaling grows one machine")
print("(more CPU/RAM/GPU) and is limited by the largest available hardware.")

=== Cloud Autoscaling for ML Inference ===

Scalability = adjust capacity to match demand while keeping latency
stable. The control loop:
  monitor metrics -> evaluate thresholds -> adjust capacity -> stabilise

Config: 100 req/s per replica, target CPU 70%, min=2, max=20 replicas

 hour   req/s  replicas    util   latency  action
--------------------------------------------------------------
    0     120         2     60%      100ms  hold
    3      60         2     30%       57ms  hold
    6     220         4     55%       89ms  scale UP
    9     780        12     65%      114ms  scale UP
   12    1180        17     69%      131ms  scale UP
   15     820        12     68%      126ms  scale DOWN
   18    1320        19     69%      131ms  scale UP
   21     520         8     65%      114ms  scale DOWN

=== Autoscaling vs Fixed Capacity (24 hours) ===

Strategy                            pod-hours      cost  SLA breaches
---------------------------------------------------------------

## Example 2: Cost Model - On-Demand vs Spot vs Serverless

This quantifies the cost decisions described above. For a 200 GPU-hour training job, spot instances save about 71% even after accounting for interruptions and checkpointing overhead. For inference, it computes the break-even point between always-on containers and serverless - below roughly 30 million requests a month serverless wins, above it a container does.

In [2]:
# Chapter 32 - Example 2: Cloud cost model - on-demand vs spot vs serverless
import numpy as np

print("=== Choosing Compute: On-Demand vs Spot vs Serverless ===\n")

# ---------- Training workload: GPU hours ----------
print("--- Workload A: TRAINING a deep model (fault-tolerant, batch) ---\n")
GPU_HOURS = 200
ON_DEMAND = 3.06          # $/GPU-hour
SPOT_DISCOUNT = 0.75      # spot is ~70-90% cheaper
SPOT = ON_DEMAND * (1 - SPOT_DISCOUNT)
INTERRUPT_RATE = 0.12     # fraction of hours lost to preemption
CHECKPOINT_OVERHEAD = 0.05

on_demand_cost = GPU_HOURS * ON_DEMAND
spot_hours = GPU_HOURS * (1 + INTERRUPT_RATE + CHECKPOINT_OVERHEAD)
spot_cost  = spot_hours * SPOT

print(f"  Job size: {GPU_HOURS} GPU-hours")
print(f"  {'Option':<34}{'rate $/h':>10}{'hours':>9}{'total':>11}")
print("  " + "-" * 64)
print(f"  {'On-demand GPU':<34}{ON_DEMAND:>10.2f}{GPU_HOURS:>9.0f}{on_demand_cost:>11.2f}")
print(f"  {'Spot / preemptible GPU':<34}{SPOT:>10.2f}{spot_hours:>9.0f}{spot_cost:>11.2f}")
print(f"\n  Saving with spot: ${on_demand_cost - spot_cost:,.2f} "
      f"({(1 - spot_cost/on_demand_cost)*100:.0f}%)")
print("  Spot requires checkpointing: interrupted work must resume, not restart.")
print("  Suitable for batch training, ETL and hyperparameter tuning;")
print("  NOT suitable for latency-critical production inference.\n")

# ---------- Inference workload: containers vs serverless ----------
print("--- Workload B: INFERENCE endpoint (latency sensitive) ---\n")
CONTAINER_HOUR = 0.096            # small always-on container instance
LAMBDA_PER_REQ = 0.0000002        # per invocation
LAMBDA_GB_SEC  = 0.0000166667     # per GB-second
MEM_GB, DURATION_S = 0.5, 0.25    # per inference call

print(f"  {'Requests/month':>16}{'Always-on container':>22}{'Serverless (FaaS)':>20}  {'Cheaper'}")
print("  " + "-" * 78)
for reqs in [10_000, 100_000, 1_000_000, 10_000_000, 50_000_000]:
    container = CONTAINER_HOUR * 24 * 30                       # always running
    serverless = reqs * LAMBDA_PER_REQ + reqs * MEM_GB * DURATION_S * LAMBDA_GB_SEC
    cheaper = "serverless" if serverless < container else "container"
    print(f"  {reqs:>16,}{'$'+format(container,'.2f'):>22}{'$'+format(serverless,'.2f'):>20}  {cheaper}")

break_even = (CONTAINER_HOUR*24*30) / (LAMBDA_PER_REQ + MEM_GB*DURATION_S*LAMBDA_GB_SEC)
print(f"\n  Break-even point: about {break_even:,.0f} requests/month")
print("  Below it, serverless is cheaper (scale-to-zero when idle).")
print("  Above it, an always-on container is cheaper and avoids cold starts.\n")

# ---------- Decision guidance ----------
print("=== Choosing the right service ===\n")
rows = [
 ("Large distributed training", "GPU/TPU nodes on SageMaker, Vertex AI, Azure ML"),
 ("Custom model serving",       "Containers on GKE / EKS / AKS or ECS-Fargate"),
 ("Lightweight inference",      "Serverless: AWS Lambda, Cloud Functions, Azure Functions"),
 ("Feature storage & serving",  "BigQuery, DynamoDB, Cosmos DB"),
 ("Cost-sensitive batch jobs",  "Spot / preemptible instances with checkpointing"),
]
print(f"  {'Need':<30}{'Recommended'}")
print("  " + "-" * 74)
for need, rec in rows:
    print(f"  {need:<30}{rec}")

print("\n  Serverless limits: execution time caps, limited memory/CPU, no GPU,")
print("  cold-start latency, and statelessness - so it cannot train large models.")

=== Choosing Compute: On-Demand vs Spot vs Serverless ===

--- Workload A: TRAINING a deep model (fault-tolerant, batch) ---

  Job size: 200 GPU-hours
  Option                              rate $/h    hours      total
  ----------------------------------------------------------------
  On-demand GPU                           3.06      200     612.00
  Spot / preemptible GPU                  0.77      234     179.01

  Saving with spot: $432.99 (71%)
  Spot requires checkpointing: interrupted work must resume, not restart.
  Suitable for batch training, ETL and hyperparameter tuning;
  NOT suitable for latency-critical production inference.

--- Workload B: INFERENCE endpoint (latency sensitive) ---

    Requests/month   Always-on container   Serverless (FaaS)  Cheaper
  ------------------------------------------------------------------------------
            10,000                $69.12               $0.02  serverless
           100,000                $69.12               $0.23  serv

## Example 3: Batch vs Streaming, and Container Reproducibility

The first half compares batch and streaming architectures on latency and throughput and simulates a traffic burst, showing how a backlog forms when arrivals exceed capacity - the exact problem autoscaling solves. The second half shows how a Dockerfile pins the whole environment so that three machines that would otherwise drift on Python, numpy and sklearn versions become identical.

In [3]:
# Chapter 32 - Example 3: Batch vs Streaming architectures + container reproducibility
import numpy as np, hashlib
np.random.seed(1)

print("=== Batch vs Streaming Architectures ===\n")
print("Batch     : process large volumes at scheduled intervals (retraining, ETL)")
print("Streaming : process records in real time as they arrive (fraud, IoT)\n")

N_EVENTS = 10_000
EVENT_RATE = 500          # events per second arriving
BATCH_INTERVAL_S = 3600   # batch job runs hourly
BATCH_THROUGHPUT = 5000   # records/sec when the batch job runs
STREAM_THROUGHPUT = 600   # records/sec continuous
STREAM_PER_REC_MS = 12

# ---- Batch ----
batch_runtime = N_EVENTS / BATCH_THROUGHPUT
avg_wait = BATCH_INTERVAL_S / 2                 # average time an event waits
batch_latency = avg_wait + batch_runtime
# ---- Streaming ----
stream_latency = STREAM_PER_REC_MS / 1000.0

print(f"Workload: {N_EVENTS:,} events arriving at {EVENT_RATE}/s\n")
print(f"  {'Metric':<34}{'Batch':>16}{'Streaming':>16}")
print("  " + "-" * 66)
print(f"  {'Processing trigger':<34}{'every 60 min':>16}{'per event':>16}")
print(f"  {'End-to-end latency':<34}{batch_latency:>13.1f} s{stream_latency*1000:>13.0f} ms")
print(f"  {'Throughput':<34}{BATCH_THROUGHPUT:>13,}/s{STREAM_THROUGHPUT:>13,}/s")
print(f"  {'Compute utilisation':<34}{'bursty':>16}{'continuous':>16}")
print(f"  {'Relative cost':<34}{'lower':>16}{'higher':>16}")

print(f"\n  Latency difference: {batch_latency/stream_latency:,.0f}x")
print("\n  Choose BATCH     : scheduled retraining, historical scoring, ETL")
print("  Choose STREAMING : fraud detection, live recommendations, IoT analytics")
print("  Tools for streaming: Pub/Sub, Kinesis, Event Hub")
print("  HYBRID is common: stream for real-time decisions, batch for retraining.\n")

# ---- Simulate a streaming queue under a traffic burst ----
print("--- Streaming queue behaviour during a burst ---\n")
arrival = np.array([300, 350, 900, 1200, 1100, 400, 320, 300])   # events/s per window
queue = 0
print(f"  {'window':>7}{'arrivals/s':>12}{'capacity/s':>12}{'queue':>8}{'latency':>11}")
print("  " + "-" * 52)
for i, a in enumerate(arrival, 1):
    queue = max(0, queue + a - STREAM_THROUGHPUT)
    lat = STREAM_PER_REC_MS + (queue / STREAM_THROUGHPUT) * 1000
    flag = "  <-- backlog" if queue > 0 else ""
    print(f"  {i:>7}{a:>12}{STREAM_THROUGHPUT:>12}{queue:>8}{lat:>9.0f}ms{flag}")
print("\n  A sustained burst above capacity builds a backlog and latency grows.")
print("  This is exactly what autoscaling (adding consumers) is meant to fix.\n")

# ---- Container reproducibility ----
print("=== Containerisation: solving 'works on my machine' ===\n")
dockerfile = """FROM python:3.10-slim
RUN pip install numpy==2.4.4 scikit-learn==1.8.0 pandas==3.0.2
COPY model.pkl /app/model.pkl
COPY serve.py  /app/serve.py
CMD ["python", "/app/serve.py"]"""
print("Dockerfile (pins the ENTIRE environment):")
for line in dockerfile.split("\n"):
    print(f"    {line}")

img = hashlib.sha256(dockerfile.encode()).hexdigest()[:16]
print(f"\n  Image digest: sha256:{img}")
print("  Same Dockerfile -> same digest -> identical environment everywhere.\n")

envs = {
  "Developer laptop (no container)": {"python": "3.11.2", "numpy": "2.1.0", "sklearn": "1.6.0"},
  "CI server (no container)"       : {"python": "3.9.16", "numpy": "2.4.4", "sklearn": "1.8.0"},
  "Production VM (no container)"   : {"python": "3.10.9", "numpy": "2.0.1", "sklearn": "1.7.1"},
}
pinned = {"python": "3.10", "numpy": "2.4.4", "sklearn": "1.8.0"}
print(f"  {'Environment':<34}{'python':>9}{'numpy':>9}{'sklearn':>10}  {'Matches pinned?'}")
print("  " + "-" * 78)
for name, e in envs.items():
    ok = all(e[k].startswith(pinned[k]) for k in pinned)
    print(f"  {name:<34}{e['python']:>9}{e['numpy']:>9}{e['sklearn']:>10}  {'yes' if ok else 'NO - drift'}")
print(f"  {'Any machine WITH the container':<34}{pinned['python']:>9}{pinned['numpy']:>9}{pinned['sklearn']:>10}  yes")

print("\n  Containers bundle code + dependencies + runtime into one image, so")
print("  CUDA/cuDNN versions, library versions and the OS layer stay identical")
print("  across training, CI, and production.")

=== Batch vs Streaming Architectures ===

Batch     : process large volumes at scheduled intervals (retraining, ETL)
Streaming : process records in real time as they arrive (fraud, IoT)

Workload: 10,000 events arriving at 500/s

  Metric                                       Batch       Streaming
  ------------------------------------------------------------------
  Processing trigger                    every 60 min       per event
  End-to-end latency                       1802.0 s           12 ms
  Throughput                                5,000/s          600/s
  Compute utilisation                         bursty      continuous
  Relative cost                                lower          higher

  Latency difference: 150,167x

  Choose BATCH     : scheduled retraining, historical scoring, ETL
  Choose STREAMING : fraud detection, live recommendations, IoT analytics
  Tools for streaming: Pub/Sub, Kinesis, Event Hub
  HYBRID is common: stream for real-time decisions, batch for ret